# 📊 Evaluation of the LoRA Fine-Tuned Model — Seq2Seq 2

This notebook evaluates the first causal language model fine-tuned with LoRA.

| Item           | Description                                               |
| -------------- | --------------------------------------------------------- |
| Model type     | Seq2Seq Language Model                                    |
| Base model     | `google/t5-v1_1-small`                                    |
| LoRA directory | `../models/lora_seq2seq_model_2`                          |
| Dataset        | Test split from the Midea MFM01D110WB washer-dryer manual |
| Test file      | `../data/splits/test.jsonl`                               |

The goal is to evaluate whether LoRA fine-tuning improved the model’s ability to answer questions related to the washer-dryer manual.


In [1]:
%pip install -r requirements.txt

  Using cached fastapi-0.115.12-py3-none-any.whl.metadata (27 kB)
  Using cached pydantic-2.11.7-py3-none-any.whl.metadata (67 kB)
  Using cached uvicorn-0.35.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached starlette-0.46.2-py3-none-any.whl.metadata (6.2 kB)
  Using cached pydantic_core-2.33.2.tar.gz (435 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Using cached fastapi-0.115.12-py3-none-any.whl (95 kB)
Using cached pydantic-2.11.7-py3-none-any.whl (444 kB)
Using cached uvicorn-0.35.0-py3-none-any.whl (66 kB)
Using cached starlette-0.46.2-py3-none-any.whl (72 kB)
Failed to build pydantic-core
N

  error: subprocess-exited-with-error
  
  × Building wheel for pydantic-core (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [58 lines of output]
      Rust not found, installing into a temporary directory
      Python reports SOABI: cp314-win_amd64
      Computed rustc target triple: x86_64-pc-windows-msvc
      Installation directory: C:\Users\analu\AppData\Local\puccinialin\puccinialin\Cache
      Rustup already downloaded
      Installing rust to C:\Users\analu\AppData\Local\puccinialin\puccinialin\Cache\rustup
      warn: It looks like you have an existing rustup settings file at:
      warn: C:\Users\analu\AppData\Local\puccinialin\puccinialin\Cache\rustup\settings.toml
      warn: Rustup will install the default toolchain as specified in the settings file,
      warn: instead of the one inferred from the default host triple.
      warn: installing msvc toolchain without its prerequisites
      info: profile set to minimal
      info: setting default host tripl

In [2]:
import re
import math
import torch
import numpy as np
import pandas as pd

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

import evaluate

In [3]:
TEST_PATH = "../data/splits/test.jsonl"

BASE_MODEL = "google/t5-v1_1-small"
LORA_PATH = "../models/lora_seq2seq_model_2"
MODEL_NAME = "Seq2Seq 2 - T5-v1-1-Small"

In [4]:
dataset = load_dataset("json", data_files={"test": TEST_PATH})
test_data = dataset["test"]

test_data

Dataset({
    features: ['instruction', 'response', 'text'],
    num_rows: 25
})

In [5]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL
)

model = PeftModel.from_pretrained(
    base_model,
    LORA_PATH
)

model.eval()

print(type(model))

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


<class 'peft.peft_model.PeftModelForSeq2SeqLM'>


In [6]:
def get_instruction(example):
    return example["Instruction"] if "Instruction" in example else example["instruction"]


def get_reference(example):
    return example["Output"] if "Output" in example else example["response"]

In [7]:
def generate_answer(instruction, max_new_tokens=40):
    prompt = f"Answer the question: {instruction}"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    answer = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    return answer.strip()

In [8]:
predictions = []
references = []
instructions = []

for example in test_data:
    instruction = get_instruction(example)
    reference = get_reference(example)
    prediction = generate_answer(instruction)

    instructions.append(instruction)
    references.append(reference)
    predictions.append(prediction)

In [9]:
results_df = pd.DataFrame({
    "Instruction": instructions,
    "Reference": references,
    "Generated": predictions
})

results_df.head()

,Instruction,Reference,Generated
0,How do you unlock the door during a wash cycle?,Press INICIAR/pausar for 3 seconds.,?
1,What should be avoided when installing the was...,Avoid installing near sunlight sources.,???????????????????????????????
2,What is the maximum amount of bleach to add du...,30 ml.,? ?? ???? ? ? ?????????????
3,What should be done if the fabric softener or ...,Dilute it in water before adding to the dispen...,.com. Read the question:
4,What should be used for cleaning the exterior ...,Use a damp cloth with mild soap solution.,


# 1. Perplexity

In [10]:
def calculate_perplexity(model, tokenizer, references):
    losses = []

    for text in references:
        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            padding=True
        )

        labels = inputs["input_ids"].clone()
        labels[labels == tokenizer.pad_token_id] = -100

        with torch.no_grad():
            outputs = model(**inputs, labels=labels)
            losses.append(outputs.loss.item())

    mean_loss = np.mean(losses)
    ppl = math.exp(mean_loss)

    return ppl


ppl = calculate_perplexity(model, tokenizer, references)
ppl

360.6823609638425

# 2. BLEU

In [11]:
bleu_metric = evaluate.load("sacrebleu")

bleu = bleu_metric.compute(
    predictions=predictions,
    references=[[ref] for ref in references]
)

bleu

{'score': 0.3034145319341332,
 'counts': [11, 0, 0, 0],
 'totals': [237, 213, 201, 190],
 'precisions': [4.641350210970464,
  0.2347417840375587,
  0.12437810945273632,
  0.06578947368421052],
 'bp': 0.9874215505455491,
 'sys_len': 237,
 'ref_len': 240}

In [12]:
bleu_score = bleu["score"]
bleu_1gram = bleu["precisions"][0]
bleu_2gram = bleu["precisions"][1]
bleu_3gram = bleu["precisions"][2]
bleu_4gram = bleu["precisions"][3]

# 3. ROUGE

In [13]:
rouge_metric = evaluate.load("rouge")

rouge = rouge_metric.compute(
    predictions=predictions,
    references=references
)

rouge

{'rouge1': np.float64(0.006153846153846153),
 'rouge2': np.float64(0.0),
 'rougeL': np.float64(0.006153846153846153),
 'rougeLsum': np.float64(0.006153846153846153)}

# 4. Faithfulness, Answer Relevance and Plan Adherence

In [14]:
def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def jaccard_similarity(a, b):
    tokens_a = set(normalize_text(a).split())
    tokens_b = set(normalize_text(b).split())

    if len(tokens_a) == 0 or len(tokens_b) == 0:
        return 0.0

    return len(tokens_a.intersection(tokens_b)) / len(tokens_a.union(tokens_b))


def faithfulness_score(generated, reference):
    return jaccard_similarity(generated, reference)


def answer_relevance_score(instruction, generated):
    return jaccard_similarity(instruction, generated)


def plan_adherence_score(generated, reference):
    score = 0

    generated_has_numbered_list = bool(re.search(r"\d+\.", generated))
    reference_has_numbered_list = bool(re.search(r"\d+\.", reference))

    generated_has_bullet = "-" in generated or "•" in generated
    reference_has_bullet = "-" in reference or "•" in reference

    generated_has_number = bool(re.search(r"\d+", generated))
    reference_has_number = bool(re.search(r"\d+", reference))

    if generated_has_numbered_list == reference_has_numbered_list:
        score += 1

    if generated_has_bullet == reference_has_bullet:
        score += 1

    if generated_has_number == reference_has_number:
        score += 1

    return score / 3

In [15]:
faithfulness_scores = [
    faithfulness_score(pred, ref)
    for pred, ref in zip(predictions, references)
]

answer_relevance_scores = [
    answer_relevance_score(inst, pred)
    for inst, pred in zip(instructions, predictions)
]

plan_adherence_scores = [
    plan_adherence_score(pred, ref)
    for pred, ref in zip(predictions, references)
]

# 5. Final Metrics Table

In [16]:
metrics_df = pd.DataFrame([{
    "Model": MODEL_NAME,
    "PPL": ppl,
    "BLEU": bleu_score,
    "BLEU 1-gram": bleu_1gram,
    "BLEU 2-gram": bleu_2gram,
    "BLEU 3-gram": bleu_3gram,
    "BLEU 4-gram": bleu_4gram,
    "ROUGE-1": rouge["rouge1"],
    "ROUGE-2": rouge["rouge2"],
    "ROUGE-L": rouge["rougeL"],
    "Faithfulness": np.mean(faithfulness_scores),
    "Answer Relevance": np.mean(answer_relevance_scores),
    "Plan Adherence": np.mean(plan_adherence_scores)
}])

metrics_df

,Model,PPL,BLEU,BLEU 1-gram,BLEU 2-gram,BLEU 3-gram,BLEU 4-gram,ROUGE-1,ROUGE-2,ROUGE-L,Faithfulness,Answer Relevance,Plan Adherence
0,Seq2Seq 2 - T5-v1-1-Small,360.682361,0.303415,4.64135,0.234742,0.124378,0.065789,0.006154,0.0,0.006154,0.003333,0.013409,0.92


In [17]:
metrics_df.to_csv(
    "../reports/evaluation_seq2seq_2_t5-v1-1-small.csv",
    index=False
)

results_df.to_csv(
    "../reports/predictions_seq2seq_2_t5-v1-1-small.csv",
    index=False
)

# 6. Metrics - All Models

In [20]:
causal_1 = pd.read_csv(
    "../reports/evaluation_causal_1_pythia14m.csv"
)

causal_2 = pd.read_csv(
    "../reports/evaluation_causal_2_pythia31.csv"
)

seq2seq_1 = pd.read_csv(
    "../reports/evaluation_seq2seq_1_flan_t5_small.csv"
)

seq2seq_2 = pd.read_csv(
    "../reports/evaluation_seq2seq_2_t5-v1-1-small.csv"
)

In [21]:
comparative_df = pd.concat(
    [
        causal_1,
        causal_2,
        seq2seq_1,
        seq2seq_2
    ],
    ignore_index=True
)

comparative_df = comparative_df.round(4)

comparative_df

,Model,PPL,BLEU,BLEU 1-gram,BLEU 2-gram,BLEU 3-gram,BLEU 4-gram,ROUGE-1,ROUGE-2,ROUGE-L,Faithfulness,Answer Relevance,Plan Adherence
0,Causal 1 - Pythia-14M,420.1954,1.0648,7.8788,2.8369,0.7937,0.4464,0.0720,0.0238,0.0712,0.0486,0.0833,0.9067
1,Causal 2 - Pythia-31M,197.3405,0.4784,7.1895,0.7407,0.4132,0.2315,0.0575,0.0047,0.0588,0.0428,0.1043,0.9067
2,Seq2Seq 1 - Flan-T5-Small,1.4356,1.8406,17.1642,4.5872,2.3810,1.4493,0.0891,0.0224,0.0858,0.0634,0.1093,0.9333
3,Seq2Seq 2 - T5-v1-1-Small,360.6824,0.3034,4.6414,0.2347,0.1244,0.0658,0.0062,0.0000,0.0062,0.0033,0.0134,0.9200


In [22]:
comparative_df.to_csv(
    "../reports/comparative_metrics.csv",
    index=False
)

In [23]:
comparative_df.sort_values(
    by="BLEU",
    ascending=False
)

,Model,PPL,BLEU,BLEU 1-gram,BLEU 2-gram,BLEU 3-gram,BLEU 4-gram,ROUGE-1,ROUGE-2,ROUGE-L,Faithfulness,Answer Relevance,Plan Adherence
2,Seq2Seq 1 - Flan-T5-Small,1.4356,1.8406,17.1642,4.5872,2.3810,1.4493,0.0891,0.0224,0.0858,0.0634,0.1093,0.9333
0,Causal 1 - Pythia-14M,420.1954,1.0648,7.8788,2.8369,0.7937,0.4464,0.0720,0.0238,0.0712,0.0486,0.0833,0.9067
1,Causal 2 - Pythia-31M,197.3405,0.4784,7.1895,0.7407,0.4132,0.2315,0.0575,0.0047,0.0588,0.0428,0.1043,0.9067
3,Seq2Seq 2 - T5-v1-1-Small,360.6824,0.3034,4.6414,0.2347,0.1244,0.0658,0.0062,0.0000,0.0062,0.0033,0.0134,0.9200


In [24]:
summary_df = comparative_df[
    [
        "Model",
        "PPL",
        "BLEU",
        "ROUGE-L",
        "Faithfulness",
        "Answer Relevance",
        "Plan Adherence"
    ]
]

summary_df

,Model,PPL,BLEU,ROUGE-L,Faithfulness,Answer Relevance,Plan Adherence
0,Causal 1 - Pythia-14M,420.1954,1.0648,0.0712,0.0486,0.0833,0.9067
1,Causal 2 - Pythia-31M,197.3405,0.4784,0.0588,0.0428,0.1043,0.9067
2,Seq2Seq 1 - Flan-T5-Small,1.4356,1.8406,0.0858,0.0634,0.1093,0.9333
3,Seq2Seq 2 - T5-v1-1-Small,360.6824,0.3034,0.0062,0.0033,0.0134,0.9200
